In [ ]:
import duckdb
import os

In [6]:
conn = duckdb.connect('warehouse.duckdb')
conn.execute("LOAD httpfs;")
conn.execute("""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

result = conn.execute("""
    SELECT DISTINCT ticker 
    FROM read_parquet('s3://staging/prices/*.parquet')
""").fetchdf()
print(result)

  ticker
0   AMZN
1   NVDA
2   TSLA
3   AAPL
4  GOOGL
5   META
6   MSFT


In [ ]:
conn = duckdb.connect('warehouse.duckdb')
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")
conn.execute(f"""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# vediamo quanti file trova
result = conn.execute("SELECT * FROM glob('s3://staging/prices/*.parquet')").fetchdf()
print(result)

                                       file
0   s3://staging/prices/AAPL_prices.parquet
1   s3://staging/prices/AMZN_prices.parquet
2  s3://staging/prices/GOOGL_prices.parquet
3   s3://staging/prices/META_prices.parquet
4   s3://staging/prices/MSFT_prices.parquet
5   s3://staging/prices/NVDA_prices.parquet
6   s3://staging/prices/TSLA_prices.parquet


In [2]:
conn = duckdb.connect('warehouse.duckdb')
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")
conn.execute(f"""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# vediamo quanti file trova
result = conn.execute("SELECT * FROM glob('s3://staging/earnings/*.parquet')").fetchdf()
print(result)

                                           file
0   s3://staging/earnings/AAPL_earnings.parquet
1   s3://staging/earnings/AMZN_earnings.parquet
2  s3://staging/earnings/GOOGL_earnings.parquet
3   s3://staging/earnings/META_earnings.parquet
4   s3://staging/earnings/MSFT_earnings.parquet
5   s3://staging/earnings/NVDA_earnings.parquet
6   s3://staging/earnings/TSLA_earnings.parquet


In [3]:
conn = duckdb.connect('warehouse.duckdb')
conn.execute("INSTALL httpfs;")
conn.execute("LOAD httpfs;")
conn.execute(f"""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

# vediamo quanti file trova
result = conn.execute("SELECT * FROM glob('s3://staging/overview/*.parquet')").fetchdf()
print(result)

                                           file
0   s3://staging/overview/AAPL_overview.parquet
1   s3://staging/overview/AMZN_overview.parquet
2  s3://staging/overview/GOOGL_overview.parquet
3   s3://staging/overview/META_overview.parquet
4   s3://staging/overview/MSFT_overview.parquet
5   s3://staging/overview/NVDA_overview.parquet
6   s3://staging/overview/TSLA_overview.parquet


In [15]:
tickers = ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
for t in tickers:
    result = conn.execute(f"""
        SELECT DISTINCT ticker, COUNT(*) as rows
        FROM read_parquet('s3://staging/prices/{t}_prices.parquet')
        GROUP BY ticker
    """).fetchdf()
    print(f"File {t}: {result.to_dict('records')}")

File AAPL: [{'ticker': 'AAPL', 'rows': 100}]
File AMZN: [{'ticker': 'MSFT', 'rows': 100}]
File GOOGL: [{'ticker': 'MSFT', 'rows': 100}]
File META: [{'ticker': 'MSFT', 'rows': 100}]
File MSFT: [{'ticker': 'MSFT', 'rows': 100}]
File NVDA: [{'ticker': 'MSFT', 'rows': 100}]
File TSLA: [{'ticker': 'MSFT', 'rows': 100}]


In [8]:
tickers = ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
for t in tickers:
    result = conn.execute(f"""
        SELECT DISTINCT ticker, COUNT(*) as rows
        FROM read_parquet('s3://staging/earnings/{t}_earnings.parquet')
        GROUP BY ticker
    """).fetchdf()
    print(f"File {t}: {result.to_dict('records')}")

File AAPL: [{'ticker': 'AAPL', 'rows': 121}]
File AMZN: [{'ticker': 'AMZN', 'rows': 116}]
File GOOGL: [{'ticker': 'GOOGL', 'rows': 87}]
File META: [{'ticker': 'META', 'rows': 57}]
File MSFT: [{'ticker': 'MSFT', 'rows': 121}]
File NVDA: [{'ticker': 'NVDA', 'rows': 109}]
File TSLA: [{'ticker': 'TSLA', 'rows': 64}]


In [9]:
tickers = ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'TSLA']
for t in tickers:
    result = conn.execute(f"""
        SELECT DISTINCT ticker, COUNT(*) as rows
        FROM read_parquet('s3://staging/overview/{t}_overview.parquet')
        GROUP BY ticker
    """).fetchdf()
    print(f"File {t}: {result.to_dict('records')}")

File AAPL: [{'ticker': 'AAPL', 'rows': 1}]
File AMZN: [{'ticker': 'AMZN', 'rows': 1}]
File GOOGL: [{'ticker': 'GOOGL', 'rows': 1}]
File META: [{'ticker': 'META', 'rows': 1}]
File MSFT: [{'ticker': 'MSFT', 'rows': 1}]
File NVDA: [{'ticker': 'NVDA', 'rows': 1}]
File TSLA: [{'ticker': 'TSLA', 'rows': 1}]


DuckDB permette connessioni multiple in lettura simultanee, mentre il blocco si verifica solo quando qualcuno tenta una connessione in scrittura mentre un'altra è già aperta


In [5]:
conn = duckdb.connect('warehouse.duckdb', read_only=True)

In [3]:
conn.execute("SHOW TABLES").fetchdf()


,name
0,stg_earnings
1,stg_overview
2,stg_prices


In [7]:
table_name = "stg_prices"  # cambia qui per esplorare un'altra tabella/vista

df = conn.execute(f"SELECT * FROM {table_name} LIMIT 20").fetchdf()
print(df.shape)
df

(20, 7)


,ticker,date,open,high,low,close,volume
0,AAPL,2026-06-18,298.110,300.57,295.620,298.01,85962201
1,AAPL,2026-06-17,300.845,302.07,294.360,295.95,42745060
2,AAPL,2026-06-16,295.245,300.48,293.970,299.24,39874404
3,AAPL,2026-06-15,294.120,297.78,291.700,296.42,45732573
4,AAPL,2026-06-12,296.030,297.14,289.620,291.13,38784789
5,AAPL,2026-06-11,293.720,297.00,289.590,295.63,42572497
6,AAPL,2026-06-10,290.740,294.75,287.380,291.58,52793266
7,AAPL,2026-06-09,300.275,300.75,287.780,290.55,70108847
8,AAPL,2026-06-08,308.739,317.40,301.170,301.54,77949082
9,AAPL,2026-06-05,312.860,315.17,307.150,307.34,65310502


In [6]:

print(conn.execute(f"SELECT distinct ticker FROM stg_prices").fetchdf())


  ticker
0   MSFT
1   AAPL


In [5]:
table_name = "stg_earnings"  # cambia qui per esplorare un'altra tabella/vista

df = conn.execute(f"SELECT * FROM {table_name} LIMIT 20").fetchdf()
print(df.shape)
df

(20, 8)


,ticker,fiscal_date_ending,reported_date,reported_eps,estimated_eps,surprise,surprise_percentage,report_time
0,AAPL,2026-03-31,2026-04-30,2.01,1.94,0.07,3.6082,post-market
1,AAPL,2025-12-31,2026-01-29,2.84,2.67,0.17,6.3670,post-market
2,AAPL,2025-09-30,2025-10-30,1.85,1.77,0.08,4.5198,post-market
3,AAPL,2025-06-30,2025-07-31,1.57,1.43,0.14,9.7902,post-market
4,AAPL,2025-03-31,2025-05-01,1.65,1.62,0.03,1.8519,post-market
5,AAPL,2024-12-31,2025-01-30,2.40,2.34,0.06,2.5641,post-market
6,AAPL,2024-09-30,2024-10-31,0.97,0.95,0.02,2.1053,post-market
7,AAPL,2024-06-30,2024-08-01,1.40,1.34,0.06,4.4776,post-market
8,AAPL,2024-03-31,2024-05-02,1.53,1.50,0.03,2.0000,post-market
9,AAPL,2023-12-31,2024-02-01,2.18,2.11,0.07,3.3175,post-market


In [6]:
table_name = "stg_overview"  # cambia qui per esplorare un'altra tabella/vista

df = conn.execute(f"SELECT * FROM {table_name} LIMIT 20").fetchdf()
print(df.shape)
df

(7, 17)


,ticker,symbol,name,exchange,currency,country,sector,industry,market_capitalization,ebitda,pe_ratio,eps,beta,week_52_high,week_52_low,dividend_yield,profit_margin
0,AAPL,AAPL,Apple Inc,NASDAQ,USD,USA,TECHNOLOGY,CONSUMER ELECTRONICS,4275929874000,159975997000,35.20,8.27,1.086,317.40,194.30,0.0035,0.2720
1,AMZN,AMZN,Amazon.com Inc,NASDAQ,USD,USA,CONSUMER CYCLICAL,INTERNET RETAIL,2566108479000,155860992000,31.64,7.54,1.444,278.56,196.00,NaN,0.1220
2,GOOGL,GOOGL,Alphabet Inc Class A,NASDAQ,USD,USA,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,4386274148000,161315996000,27.48,13.09,1.237,408.37,161.54,0.0023,0.3790
3,META,META,Meta Platforms Inc.,NASDAQ,USD,USA,COMMUNICATION SERVICES,INTERNET CONTENT & INFORMATION,1439235178000,109308002000,20.62,27.50,1.229,794.38,520.26,0.0037,0.3280
4,MSFT,MSFT,Microsoft Corporation,NASDAQ,USD,USA,TECHNOLOGY,SOFTWARE - INFRASTRUCTURE,2902586556000,184457003000,23.26,16.80,1.103,551.05,355.51,0.0091,0.3930
5,NVDA,NVDA,NVIDIA Corporation,NASDAQ,USD,USA,TECHNOLOGY,SEMICONDUCTORS,4969906831000,165514002000,31.42,6.53,2.202,236.26,141.84,0.0002,0.6300
6,TSLA,TSLA,Tesla Inc,NASDAQ,USD,USA,CONSUMER CYCLICAL,AUTO MANUFACTURERS,1526438822000,11094000000,369.48,1.10,1.798,498.83,288.77,NaN,0.0395


In [12]:
print(conn.execute("SELECT * FROM int_company_metrics").fetchdf())

  ticker                   name                  sector  \
0   AAPL              Apple Inc              TECHNOLOGY   
1   AMZN         Amazon.com Inc       CONSUMER CYCLICAL   
2  GOOGL   Alphabet Inc Class A  COMMUNICATION SERVICES   
3   META    Meta Platforms Inc.  COMMUNICATION SERVICES   
4   MSFT  Microsoft Corporation              TECHNOLOGY   
5   NVDA     NVIDIA Corporation              TECHNOLOGY   
6   TSLA              Tesla Inc       CONSUMER CYCLICAL   

                         industry  market_capitalization  pe_ratio    eps  \
0            CONSUMER ELECTRONICS          4275929874000     35.20   8.27   
1                 INTERNET RETAIL          2566108479000     31.64   7.54   
2  INTERNET CONTENT & INFORMATION          4386274148000     27.48  13.09   
3  INTERNET CONTENT & INFORMATION          1439235178000     20.62  27.50   
4       SOFTWARE - INFRASTRUCTURE          2902586556000     23.26  16.80   
5                  SEMICONDUCTORS          4969906831000     31.4

In [16]:
print(conn.execute("SELECT * FROM dim_company").fetchdf())

  ticker                   name                  sector  \
0   AAPL              Apple Inc              TECHNOLOGY   
1   AMZN         Amazon.com Inc       CONSUMER CYCLICAL   
2  GOOGL   Alphabet Inc Class A  COMMUNICATION SERVICES   
3   META    Meta Platforms Inc.  COMMUNICATION SERVICES   
4   MSFT  Microsoft Corporation              TECHNOLOGY   
5   NVDA     NVIDIA Corporation              TECHNOLOGY   
6   TSLA              Tesla Inc       CONSUMER CYCLICAL   

                         industry  market_capitalization  pe_ratio    eps  \
0            CONSUMER ELECTRONICS          4275929874000     35.20   8.27   
1                 INTERNET RETAIL          2566108479000     31.64   7.54   
2  INTERNET CONTENT & INFORMATION          4386274148000     27.48  13.09   
3  INTERNET CONTENT & INFORMATION          1439235178000     20.62  27.50   
4       SOFTWARE - INFRASTRUCTURE          2902586556000     23.26  16.80   
5                  SEMICONDUCTORS          4969906831000     31.4

In [17]:
print(conn.execute("SELECT * FROM fact_daily_prices LIMIT 20").fetchdf())

   ticker       date     open    high      low   close    volume
0    AAPL 2026-06-12  296.030  297.14  289.620  291.13  38784789
1    AAPL 2026-06-11  293.720  297.00  289.590  295.63  42572497
2    AAPL 2026-06-10  290.740  294.75  287.380  291.58  52793266
3    AAPL 2026-06-09  300.275  300.75  287.780  290.55  70108847
4    AAPL 2026-06-08  308.739  317.40  301.170  301.54  77949082
5    AAPL 2026-06-05  312.860  315.17  307.150  307.34  65310502
6    AAPL 2026-06-04  313.230  313.54  309.650  311.23  44869134
7    AAPL 2026-06-03  314.175  316.94  308.850  310.26  50836705
8    AAPL 2026-06-02  307.460  315.45  306.685  315.20  44534716
9    AAPL 2026-06-01  309.625  310.94  305.020  306.31  48849933
10   AAPL 2026-05-29  311.775  315.00  309.530  312.06  70026752
11   AAPL 2026-05-28  310.680  312.80  309.570  312.51  48220390
12   AAPL 2026-05-27  308.330  313.26  308.300  310.85  50430919
13   AAPL 2026-05-26  309.560  311.82  307.670  308.33  48000493
14   AAPL 2026-05-22  306

In [3]:
print(conn.execute("SELECT distinct ticker FROM fact_daily_prices").fetchdf())

  ticker
0   MSFT
1   AAPL


In [22]:
print(conn.execute("SELECT * FROM fact_earnings LIMIT 20").fetchdf())

   ticker fiscal_date_ending reported_date  reported_eps  estimated_eps  \
0    AAPL         2026-03-31    2026-04-30          2.01           1.94   
1    AAPL         2025-12-31    2026-01-29          2.84           2.67   
2    AAPL         2025-09-30    2025-10-30          1.85           1.77   
3    AAPL         2025-06-30    2025-07-31          1.57           1.43   
4    AAPL         2025-03-31    2025-05-01          1.65           1.62   
5    AAPL         2024-12-31    2025-01-30          2.40           2.34   
6    AAPL         2024-09-30    2024-10-31          0.97           0.95   
7    AAPL         2024-06-30    2024-08-01          1.40           1.34   
8    AAPL         2024-03-31    2024-05-02          1.53           1.50   
9    AAPL         2023-12-31    2024-02-01          2.18           2.11   
10   AAPL         2023-09-30    2023-11-02          1.46           1.39   
11   AAPL         2023-06-30    2023-08-03          1.26           1.19   
12   AAPL         2023-03

 ### Verifica che i mart abbiano dati aggiornati e coerenti

In [4]:
# conta righe per ogni mart
for table in ['dim_company', 'fact_daily_prices', 'fact_earnings']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} righe")

# verifica che ci siano tutti e 7 i ticker
print(conn.execute("SELECT DISTINCT ticker FROM dim_company").fetchdf())

# verifica la data più recente nei prezzi (deve essere recente, non vecchia)
print(conn.execute("SELECT MAX(date) FROM fact_daily_prices").fetchdf())

dim_company: 7 righe
fact_daily_prices: 700 righe
fact_earnings: 675 righe
  ticker
0   AAPL
1   AMZN
2   TSLA
3  GOOGL
4   MSFT
5   META
6   NVDA
   max(date)
0 2026-06-18


In [10]:
conn.close() # chiude la connessione al database DuckDB